# Ensemble Learning Demo: StumpBaggingClassifier on Spambase Dataset

In this notebook, we evaluate the performance of a custom implementation of
**StumpBaggingClassifier** on a high-dimensional real-world dataset.

We use the Spambase dataset from the UCI repository.

We compare:
- A single decision stump (baseline weak learner)
- A bagging ensemble of decision stumps

This dataset is especially challenging for weak models, making it ideal for
demonstrating the power of bagging.

In [3]:
import numpy as np
import pandas as pd

from rice_ml import DecisionTreeClassifier, StumpBaggingClassifier
from rice_ml import train_test_split, StandardScaler

## Loading the Spambase Dataset

We load the dataset directly from the UCI repository.

Each row represents an email, and features are word/frequency statistics.
The target indicates whether the email is spam (1) or not (0).

In [4]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/spambase/spambase.data"

df = pd.read_csv(url, header=None)

df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57
0,0.00,0.64,0.64,0.0,0.32,0.00,0.00,0.00,0.00,0.00,0.00,0.64,0.00,0.00,0.00,0.32,0.00,1.29,1.93,0.00,0.96,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.00,0.00,0.0,0.0,0.00,0.000,0.0,0.778,0.000,0.000,3.756,61,278,1
1,0.21,0.28,0.50,0.0,0.14,0.28,0.21,0.07,0.00,0.94,0.21,0.79,0.65,0.21,0.14,0.14,0.07,0.28,3.47,0.00,1.59,0.0,0.43,0.43,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.07,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.00,0.00,0.0,0.0,0.00,0.132,0.0,0.372,0.180,0.048,5.114,101,1028,1
2,0.06,0.00,0.71,0.0,1.23,0.19,0.19,0.12,0.64,0.25,0.38,0.45,0.12,0.00,1.75,0.06,0.06,1.03,1.36,0.32,0.51,0.0,1.16,0.06,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.06,0.0,0.0,0.12,0.0,0.06,0.06,0.0,0.0,0.01,0.143,0.0,0.276,0.184,0.010,9.821,485,2259,1
3,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,0.31,0.31,0.31,0.00,0.00,0.31,0.00,0.00,3.18,0.00,0.31,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.00,0.00,0.0,0.0,0.00,0.137,0.0,0.137,0.000,0.000,3.537,40,191,1
4,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,0.31,0.31,0.31,0.00,0.00,0.31,0.00,0.00,3.18,0.00,0.31,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.00,0.00,0.0,0.0,0.00,0.135,0.0,0.135,0.000,0.000,3.537,40,191,1


## Data Preparation

We split the dataset into:
- Features (word frequency + character frequency features)
- Labels (spam vs not spam)

In [5]:
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

X.shape, y.shape

((4601, 57), (4601,))

### Train-Test Split

We evaluate generalization performance using a held-out test set.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

### Feature Scaling

Although tree-based models do not strictly require scaling,
we standardize features for stability and consistency.

In [7]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Baseline: Single Decision Stump

We train a single depth-1 decision tree to establish a weak baseline.

In [8]:
stump = DecisionTreeClassifier(max_depth=1)
stump.fit(X_train, y_train)

baseline_acc = stump.score(X_test, y_test)
baseline_acc

np.float64(0.7801911381407471)

## StumpBaggingClassifier

We now train an ensemble of decision stumps using bootstrap aggregation.

Each stump sees a different subset of the data, and predictions are combined
using majority voting.

In [9]:
model = StumpBaggingClassifier(
    n_models=30,
    seed=42
)

model.fit(X_train, y_train)

bagging_acc = model.score(X_test, y_test)
bagging_acc

np.float64(0.7845351867940921)

## Results Comparison

We compare accuracy between:
- Single decision stump
- Bagging ensemble of stumps

In [10]:
print("Baseline Stump Accuracy:", baseline_acc)
print("Bagging Accuracy:", bagging_acc)
print("Improvement:", bagging_acc - baseline_acc)

Baseline Stump Accuracy: 0.7801911381407471
Bagging Accuracy: 0.7845351867940921
Improvement: 0.004344048653344923


## Key Takeaways

- It doesn't appear that bagging had a significant improvement in performance by reducing variance
- Nevertheless, normally, High-dimensional noisy datasets strongly benefit from ensemble methods